# Notebook 01 — Data Collection & Enrichment

**Goal:** Load the Last.fm Dataset 1K baseline, fetch updated 5-year scrobble histories via the Last.fm API, and enrich with Spotify genre and audio feature data.

**Inputs:**
- `data/raw/userid-timestamp-artid-artname-traid-traname.tsv` (2.53 GB)
- `data/raw/userid-profile.tsv` (38 KB)
- Last.fm API credentials in `.env`
- Spotify API credentials in `.env`

**Outputs (all in `data/processed/`):**
- `scrobbles_baseline.parquet` — original 2009 dataset
- `scrobbles_updated.parquet` — merged baseline + last-5-year API data
- `profiles.parquet` — user demographics
- `artist_genres.parquet` — Spotify genre lookup table
- `audio_features.parquet` — Spotify audio features for top tracks

> **Note:** This notebook only needs to run once. All subsequent notebooks read from the saved Parquet files.

In [ ]:
import logging
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)

## 1. Verify environment and credentials

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')

required_vars = ['LASTFM_API_KEY', 'LASTFM_API_SECRET', 'SPOTIFY_CLIENT_ID', 'SPOTIFY_CLIENT_SECRET']
for var in required_vars:
    val = os.environ.get(var, '')
    status = '✓' if val else '✗ MISSING'
    print(f'  {var}: {status}')

## 2. Load baseline dataset (Last.fm Dataset 1K)

The large TSV is read in chunks to manage memory.

In [ ]:
from src.data.loader import load_scrobbles, load_profiles, load_config

cfg = load_config('../configs/config.yaml')

baseline_tsv = Path('../') / cfg['paths']['raw_scrobbles']
profiles_tsv = Path('../') / cfg['paths']['raw_profiles']

print(f'Baseline TSV exists: {baseline_tsv.exists()} ({baseline_tsv})')
print(f'Profiles TSV exists: {profiles_tsv.exists()} ({profiles_tsv})')

In [ ]:
# Load profiles (small — fast)
profiles = load_profiles(
    profiles_tsv,
    save_parquet='../data/processed/profiles.parquet'
)
print(f'Profiles: {len(profiles)} users')
profiles.head()

In [ ]:
# Demographics — three separate figures, each saved as EPS + PNG.
# Gender is filtered to exclude blank/NA entries so no spurious NA bar appears.
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

figures_dir = Path('../outputs/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

# ── Gender Distribution (NA / blank excluded) ────────────────────────────────
gender_series = profiles['gender'].dropna()
gender_series = gender_series[gender_series.str.strip() != '']
# Also remove any row where gender equals the column header (TSV header bleed-through)
gender_series = gender_series[~gender_series.str.lower().isin(['gender', 'n/a', 'na'])]
n_na_gender   = len(profiles) - len(gender_series)

gender_counts = gender_series.value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
gender_counts.plot(kind='bar', ax=ax, color=['#636EFA', '#EF553B', '#00CC96'], edgecolor='white')
ax.set_title('Gender Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Gender')
ax.set_ylabel('Number Of Users')
ax.tick_params(axis='x', rotation=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(figures_dir / 'demographics_gender.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'demographics_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Gender counts: {dict(gender_counts)}  |  Users with no gender recorded: {n_na_gender}")

# ── Age Distribution ──────────────────────────────────────────────────────────
age_series = profiles['age'].dropna()
age_series  = age_series[(age_series >= 10) & (age_series <= 100)]  # plausible range

fig, ax = plt.subplots(figsize=(6, 4))
age_series.hist(ax=ax, bins=20, color='#636EFA', edgecolor='white')
ax.set_title('Age Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Age (Years)')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'demographics_age.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'demographics_age.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Age: mean={age_series.mean():.1f}, median={age_series.median():.1f}, "
      f"range=[{age_series.min():.0f}, {age_series.max():.0f}]  |  "
      f"Users with no age recorded: {profiles['age'].isna().sum()}")

# ── Top 10 Countries ──────────────────────────────────────────────────────────
country_series = profiles['country'].dropna()
country_series = country_series[country_series.str.strip() != '']
country_series = country_series[~country_series.str.lower().isin(['country', 'n/a', 'na'])]

fig, ax = plt.subplots(figsize=(8, 5))
top_countries = country_series.value_counts().head(10).sort_values()
top_countries.plot(kind='barh', ax=ax, color='#636EFA', edgecolor='white')
ax.set_title('Top 10 Countries By User Count', fontsize=13, fontweight='bold')
ax.set_xlabel('Number Of Users')
ax.set_ylabel('Country')
for bar in ax.patches:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{int(bar.get_width())}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(figures_dir / 'demographics_countries.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'demographics_countries.png', dpi=150, bbox_inches='tight')
plt.show()
n_no_country = profiles['country'].isna().sum() + (profiles['country'].fillna('').str.strip() == '').sum()
print(f"Users with no country recorded: {n_no_country}")

In [ ]:
# Load baseline scrobbles (takes a few minutes for the 2.53 GB file)
if baseline_tsv.exists():
    baseline = load_scrobbles(
        baseline_tsv,
        chunksize=500_000,
        save_parquet='../data/processed/scrobbles_baseline.parquet'
    )
    print(f'Baseline: {len(baseline):,} scrobbles, {baseline["userid"].nunique()} users')
    print(f'Date range: {baseline["timestamp"].min()} → {baseline["timestamp"].max()}')
    baseline.head()
else:
    print('Baseline TSV not found — will use API data only.')
    baseline = None

## 3. Scrobble Data Note

The Last.fm Dataset 1K uses **anonymised usernames** (`user_000001` … `user_000993`). These pseudonyms do not map to real Last.fm accounts, so live API lookups always return 404. The baseline TSV (19 M scrobbles, 992 users, 2005–2013) is the complete dataset and is used directly.


In [ ]:
# The Last.fm Dataset 1K uses anonymised usernames (user_000001, user_000002, ...)
# These are pseudonyms — they do not correspond to real Last.fm accounts, so
# API lookups always return 404. The 19 M baseline scrobbles already loaded
# above are the complete dataset; no API fetch is needed.
updated = None
print('Baseline scrobbles cover the full dataset — skipping Live API fetch.')

In [ ]:
# API fetch skipped: anonymised userids cannot be resolved on Last.fm.
# scrobbles will be built from the baseline TSV only (cell below).
print('No API fetch required — proceeding with baseline data.')

In [ ]:
# Merge baseline + updated, deduplicate
import pandas as pd

frames = [f for f in [baseline, updated] if f is not None and len(f) > 0]
scrobbles = pd.concat(frames, ignore_index=True)
scrobbles = scrobbles.drop_duplicates(
    subset=['userid', 'timestamp', 'artist_name', 'track_name']
).sort_values(['userid', 'timestamp']).reset_index(drop=True)

scrobbles.to_parquet('../data/processed/scrobbles_updated.parquet', index=False)
print(f'Merged & saved: {len(scrobbles):,} scrobbles, {scrobbles["userid"].nunique()} users')

## 4. Enrich with Spotify genres

In [ ]:
from src.data.spotify_client import build_client, fetch_artist_genres

sp = build_client()
unique_artists = scrobbles['artist_name'].dropna().unique().tolist()
print(f'Unique artists to look up: {len(unique_artists):,}')

In [ ]:
# Fetches genres with persistent cache — safe to re-run
artist_genres = fetch_artist_genres(
    sp,
    unique_artists,
    cache_path='../data/processed/spotify_artist_cache.parquet',
    request_delay=0.1,
)

artist_genres.to_parquet('../data/processed/artist_genres.parquet', index=False)
found = artist_genres['spotify_artist_id'].notna().sum()
print(f'Artists matched on Spotify: {found:,} / {len(artist_genres):,}')

## 5. Enrich with Spotify audio features

In [ ]:
from src.data.spotify_client import fetch_audio_features

# Top-N most-played tracks per user — covers the bulk of each user's listening.
# The cache makes this cell safe to re-run; only uncached tracks are fetched.
TOP_TRACKS_PER_USER = 20

track_sample = (
    scrobbles
    .groupby(['userid', 'artist_name', 'track_name'])
    .size().reset_index(name='play_count')
    .sort_values(['userid', 'play_count'], ascending=[True, False])
    .groupby('userid').head(TOP_TRACKS_PER_USER)
)
track_pairs = list(set(zip(track_sample['artist_name'], track_sample['track_name'])))
print(f'Unique (artist, track) pairs to enrich: {len(track_pairs):,}')
print(f'(Top {TOP_TRACKS_PER_USER} tracks × {scrobbles["userid"].nunique()} users before deduplication)')

In [ ]:
import pandas as pd
from pathlib import Path

AUDIO_CACHE = '../data/processed/spotify_audio_features_cache.parquet'
AUDIO_OUT   = '../data/processed/audio_features.parquet'

# Check how many tracks are already cached before fetching
if Path(AUDIO_CACHE).exists():
    existing = pd.read_parquet(AUDIO_CACHE)
    print(f'Cache already contains {len(existing):,} tracks — only new ones will be fetched.')
else:
    print('No cache found — fetching all tracks from scratch.')

audio_features = fetch_audio_features(
    sp,
    track_pairs,
    cache_path=AUDIO_CACHE,
    request_delay=0.1,
)

audio_features.to_parquet(AUDIO_OUT, index=False)

# ── Match-rate report ───────────────────────────────────────────────────────
total_attempted   = len(audio_features)
matched_spotify   = audio_features['track_spotify_id'].notna().sum()
has_features      = audio_features['danceability'].notna().sum()

# Scrobble-level coverage: fraction of play events that now have audio features
def _key(a, t): return f"{str(a).strip().lower()}|||{str(t).strip().lower()}"
scrobble_keys = scrobbles.apply(lambda r: _key(r['artist_name'], r['track_name']), axis=1)
enriched_keys = set(audio_features.loc[audio_features['danceability'].notna(), 'track_key'])
scrobble_coverage = scrobble_keys.isin(enriched_keys).mean()

print('\n── Spotify Audio-Feature Match Rate ──────────────────────────────────')
print(f'  Unique (artist, track) pairs attempted : {total_attempted:>8,}')
print(f'  Resolved to a Spotify track ID         : {matched_spotify:>8,}  ({matched_spotify/total_attempted*100:.1f}%)')
print(f'  Returned audio feature values          : {has_features:>8,}  ({has_features/total_attempted*100:.1f}%)')
print(f'  Scrobble-level coverage (play events)  : {scrobble_coverage*100:>7.1f}%')
print('──────────────────────────────────────────────────────────────────────')

audio_features.head()

## Summary

All enriched data is saved to `data/processed/`. Proceed to **Notebook 02** for feature engineering.